In [38]:
%pip install pennylane strawberryfields pennylane-sf scipy==1.10.1




  Using cached PennyLane_SF-0.29.1-py3-none-any.whl.metadata (7.0 kB)
ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Python >=3.8,<3.11; 1.8.0rc3 Requires-Python >=3.8,<3.11; 1.8.0rc4 Requires-Python >=3.8,<3.11; 1.8.1 Requires-Python >=3.8,<3.11; 1.9.0 Requires-Python >=3.8,<3.12; 1.9.0rc1 Requires-Python >=3.8,<3.12; 1.9.0rc2 Requires-Python >=3.8,<3.12; 1.9.0rc3 Requires-Python >=3.8,<3.12; 1.9.1 Requires-Python >=3.8,<3.12
ERROR: Could not find 

config

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import pennylane as qml

# =============================================================
# CONFIG
# =============================================================

N_MODES    = 2       # number of CV modes in the reservoir
WINDOW     = 24      # length of sliding window (time steps)
DEPTH      = 1       # reservoir layer depth per time step
SEED_FEATS = 42      # seed for random projection matrix
SEED_RES   = 123     # seed for reservoir parameters

DISP_SCALE = 0.05    # input displacement scaling
KERR_SCALE = 0.02    # Kerr strength scaling
CUTOFF_DIM = 4       # Fock cutoff dimension (memory/performance tradeoff)

print("Configuration:")
print(f"N_MODES    : {N_MODES}")
print(f"WINDOW     : {WINDOW}")
print(f"DEPTH      : {DEPTH}")
print(f"CUTOFF_DIM : {CUTOFF_DIM}")

# =============================================================
# 1. LOAD DATA
# =============================================================

DATA_PATH = Path("QUAC 605 Project Data_processed.xlsx - 3 month_processed.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Place the processed dataset beside this notebook: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()





Features

In [31]:
# =============================================================
# 1b. DEFINE FEATURES AND TARGET, BUILD X_raw AND y_raw
# =============================================================

feature_cols = [
    "EMBEDDED_WIND_GENERATION",
    "EMBEDDED_SOLAR_GENERATION",
    "PUMP_STORAGE_PUMPING",
    "SCOTTISH_TRANSFER",
    "NEMO_FLOW",
    "ELECLINK_FLOW",
    "Temperature",
    "period_sin",
    "period_cos",
]

target_col = "ENGLAND_WALES_DEMAND"

# We ALREADY have period_sin / period_cos in the CSV, so no need to rebuild.
# Just drop any rows with NaNs in features or target (safety).
df = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

# Build raw matrices
X_raw = df[feature_cols].values.astype(float)
y_raw = df[target_col].values.astype(float)

print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)


X_raw shape: (4318, 9)
y_raw shape: (4318,)


Normalization / Random projection

In [32]:
# =============================================================
# 2. NORMALIZE X AND y
# =============================================================

X_scaler = StandardScaler()
X = X_scaler.fit_transform(X_raw)

y_scaler = StandardScaler()
y = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

T_total, n_features = X.shape
print("n_features:", n_features)

# =============================================================
# 3. RANDOM PROJECTION INTO CV MODES (Z)
# =============================================================

rng_feats = np.random.default_rng(seed=SEED_FEATS)
W_enc = rng_feats.normal(loc=0.0, scale=0.5, size=(N_MODES, n_features))
print("W_enc shape:", W_enc.shape)

def encode_with_projection(X, W):
    return (W @ X.T).T  # (T, N_MODES)

Z = encode_with_projection(X, W_enc)
print("Z shape:", Z.shape)


n_features: 9
W_enc shape: (2, 9)
Z shape: (4318, 2)


Sliding Window

In [33]:
# =============================================================
# 4. BUILD SLIDING WINDOWS OVER Z
# =============================================================

def build_Z_windows(Z, y, window_size):
    """
    Z: (T, N_MODES)
    y: (T,)
    Returns:
      Z_windows: (num_samples, window_size, N_MODES)
      y_targets: (num_samples,)
    Each window uses [t-window_size, ..., t-1] to predict y[t].
    """
    Z_windows = []
    y_targets = []
    T = Z.shape[0]
    for t in range(window_size, T):
        Z_win = Z[t - window_size : t]  # (window_size, N_MODES)
        Z_windows.append(Z_win)
        y_targets.append(y[t])
    return np.array(Z_windows), np.array(y_targets)

Z_windows, y_targets = build_Z_windows(Z, y, WINDOW)

print("Z_windows shape:", Z_windows.shape)
print("y_targets shape:", y_targets.shape)

num_samples = Z_windows.shape[0]


Z_windows shape: (4294, 24, 2)
y_targets shape: (4294,)


Kerr CV-QRC

In [34]:
# =============================================================
# 5. DEFINE CV-QRC CIRCUIT WITH KERR NONLINEARITY
# =============================================================

dev = qml.device("strawberryfields.fock", wires=N_MODES, cutoff_dim=CUTOFF_DIM)
print(f"Device ready: {N_MODES} modes, cutoff_dim={CUTOFF_DIM}")

rng_res = np.random.default_rng(seed=SEED_RES)
phi      = rng_res.normal(loc=0.0, scale=np.pi,     size=(DEPTH, N_MODES))
theta_bs = rng_res.normal(loc=0.0, scale=np.pi/4,   size=(DEPTH, max(N_MODES-1, 1)))
phi_bs   = rng_res.normal(loc=0.0, scale=2*np.pi,   size=(DEPTH, max(N_MODES-1, 1)))
kappa    = rng_res.normal(loc=0.0, scale=KERR_SCALE, size=(DEPTH, N_MODES))

@qml.qnode(dev)
def qrc_circuit_window_single(z_flat, meas_mode):
    """
    CV-QRC with Kerr nonlinearity for a full window of length WINDOW,
    returning <X> on a single mode.

    z_flat: 1D array of length WINDOW * N_MODES
    meas_mode: integer in [0, N_MODES-1]
    """
    z_window = qml.math.reshape(z_flat, (WINDOW, N_MODES))

    for t in range(WINDOW):
        z_t = z_window[t]

        # Input encoding
        for i in range(N_MODES):
            qml.Displacement(DISP_SCALE * z_t[i], 0.0, wires=i)

        # Reservoir layers
        for d in range(DEPTH):
            # Phase shifts
            for i in range(N_MODES):
                qml.Rotation(phi[d, i], wires=i)

            # Beam splitters
            if N_MODES > 1:
                for i in range(N_MODES - 1):
                    qml.Beamsplitter(theta_bs[d, i], phi_bs[d, i], wires=[i, i+1])

            # Kerr nonlinearity
            for i in range(N_MODES):
                qml.Kerr(kappa[d, i], wires=i)

    return qml.expval(qml.X(meas_mode))


DeviceError: Device strawberryfields.fock does not exist. Make sure the required plugin is installed.

Compute hidden state

In [ ]:
# =============================================================
# 6. COMPUTE HIDDEN STATES FOR ALL WINDOWS
# =============================================================

def compute_hidden_states_from_windows(Z_windows):
    num_samples = Z_windows.shape[0]
    H = np.zeros((num_samples, N_MODES))
    for i in range(num_samples):
        z_win_flat = Z_windows[i].reshape(-1)
        for m in range(N_MODES):
            H[i, m] = qrc_circuit_window_single(z_win_flat, m)
        if (i + 1) % 10 == 0:
            print(f"Computed window {i+1}/{num_samples}", end="\r")
    print()
    return H

H = compute_hidden_states_from_windows(Z_windows)
print("H shape:", H.shape)

# Quick sanity check
print("\nSample Kerr CV-QRC features + targets:")
for i in range(min(5, H.shape[0])):
    print(f"Sample {i:03d} | y = {y_targets[i]:8.3f} | H = {H[i]}")


Ridge regression

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def ridge_forecast_from_H(H, y_targets, y_scaler, df, WINDOW, train_frac=0.8, alpha=1.0):
    num_samples = H.shape[0]
    train_size  = int(train_frac * num_samples)

    X_train = H[:train_size]
    X_test  = H[train_size:]
    y_train = y_targets[:train_size]
    y_test  = y_targets[train_size:]

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)

    y_test_raw      = y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_pred_test_raw = y_scaler.inverse_transform(y_pred_test.reshape(-1, 1)).ravel()

    mae  = mean_absolute_error(y_test_raw, y_pred_test_raw)
    rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_test_raw))
    r2   = r2_score(y_test_raw, y_pred_test_raw)

    print("\n=== Kerr CV-QRC + Ridge Regression Performance (Test Set) ===")
    print(f"MAE  : {mae:,.3f}")
    print(f"RMSE : {rmse:,.3f}")
    print(f"R^2  : {r2:,.4f}")

    start_t = WINDOW + train_size
    end_t   = WINDOW + num_samples
    time_index = df.index[start_t:end_t]

    results_df = pd.DataFrame({
        "time_index"      : time_index,
        "true_demand"     : y_test_raw,
        "predicted_demand": y_pred_test_raw,
    })

    if "SETTLEMENT_DATE" in df.columns:
        results_df["SETTLEMENT_DATE"] = df.loc[time_index, "SETTLEMENT_DATE"].values
    if "SETTLEMENT_PERIOD" in df.columns:
        results_df["SETTLEMENT_PERIOD"] = df.loc[time_index, "SETTLEMENT_PERIOD"].values

    print("\n=== Sample of Forecasts (Last 10 Rows) ===")
    print(results_df.tail(10))

    return model, results_df, {"MAE": mae, "RMSE": rmse, "R2": r2}

# Run the full classical readout
model, results_df, metrics = ridge_forecast_from_H(H, y_targets, y_scaler, df, WINDOW)
